# C2 - REAL-LIFE STRESS FROM WORK-SHIFT HEART SIGNALS (step 12)

**Status: PRELIMINARY, public data. Estimator for one latent state (stress), not the capacity label.**

Question: in real working life, when a person reports stress, how do their heart rate and movement look compared with their own usual, and can a small model tell reported stress from reported no-stress for a person it has never seen?

Data: "Multimodal sensor dataset for continuous stress detection of nurses in a hospital" (Hosseini et al., Scientific Data 2022, CC0). 15 working nurses, about 1,350 hours of real shifts, wrist Empatica E4 (HR 1 Hz, movement 32 Hz, EDA and skin temperature 4 Hz, inter-beat intervals). Stress labels come from the nurse's own phone survey after an event (level 0 = no stress, 1, 2 = high). Events with no level are dropped. This data set is used on its own and never merged with other data.

**Important limits (state them in the report):**
- The stress labels are **self-reported by the person, after the event**. They are used here only as the training and test labels of a public data set. The product itself never asks for them.
- Real-life labels are noisy, so scores will be lower than in lab studies.
- Nurses, not students. Heart-rate behaviour at work is the point, but the group is different from the target users.

Rules fixed before looking at results:
1. People are never split between train and test (participant-grouped cross-validation). The participant code is never a feature.
2. Windows: 60 seconds, 30 second step, taken inside each reported event interval, at most 10 evenly spread windows per event so long events do not dominate.
3. Main target: **level 2 (high stress) vs level 0 (no stress)**. Second target: level 1 or 2 vs level 0.
4. **Movement is a feature**, so that a raised heart rate from walking is not read as stress.
5. Personal baseline = each nurse's own median heart rate during low-movement minutes across all their recordings (no labels used). Population baseline = the median of the nurses' baselines.
6. Feature sets: A heart rate + movement (raw), B heart rate relative to the person's own baseline + movement, C = B plus heart-rate variability from inter-beat intervals, D = C plus EDA and temperature (lab ceiling, not on phones).
7. Shuffled-label control. "Clearly above chance" only if `mean AUC > max(shuffled mean, 0.5) + 2 x shuffled std`.
8. Report accuracy, precision, recall, F1, balanced accuracy, AUC and the confusion matrix, never accuracy alone.

## Cell 1 - Paths and per-second tables
Each session folder holds E4 files. They are turned into one table per second (heart rate, movement, EDA, temperature) and a list of inter-beat intervals. The tables are cached on disk.

In [ ]:
import os, sys, io, zipfile, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

BASE = Path(os.environ.get('C2_NURSE_DIR', 'data/nurses'))
OUT_DIR = Path(os.environ.get('C2_OUT_DIR', 'outputs')); OUT_DIR.mkdir(exist_ok=True, parents=True)
EX = BASE / 'extracted'
if not EX.exists():
    EX.mkdir()
    z = zipfile.ZipFile(BASE / 'Stress_dataset.zip')
    for name in z.namelist():
        if name.endswith('.zip'):
            pid, fn = name.split('/')[0], name.split('/')[-1][:-4]
            dest = EX / pid / fn; dest.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(io.BytesIO(z.read(name))) as zz:
                zz.extractall(dest)
CACHE = BASE / 'seconds'; CACHE.mkdir(exist_ok=True)

def read_e4(p, ncol=1):
    with open(p) as f:
        t0 = float(f.readline().split(',')[0]); hz = float(f.readline().split(',')[0])
    d = pd.read_csv(p, skiprows=2, header=None)
    return t0, hz, d.apply(pd.to_numeric, errors='coerce').to_numpy()

def per_second(sess):
    """one row per second: hr, acc_mean (movement level), acc_sd (movement variability), eda, temp"""
    try:
        t0h, _, hr = read_e4(sess / 'HR.csv'); hr = hr.ravel()
        t0a, hza, acc = read_e4(sess / 'ACC.csv')
        t0e, hze, eda = read_e4(sess / 'EDA.csv'); eda = eda.ravel()
        t0t, hzt, tmp = read_e4(sess / 'TEMP.csv'); tmp = tmp.ravel()
    except Exception:
        return None, None
    mag = np.sqrt((acc.astype(float) ** 2).sum(axis=1)) / 64.0
    n_a = int(len(mag) // hza); m = mag[:n_a * int(hza)].reshape(n_a, int(hza))
    a = pd.Series(m.mean(axis=1), index=(t0a + np.arange(n_a)).astype(int))
    s = pd.Series(m.std(axis=1), index=a.index)
    h = pd.Series(hr, index=(t0h + np.arange(len(hr))).astype(int))
    n_e = int(len(eda) // hze); e = pd.Series(eda[:n_e * int(hze)].reshape(n_e, int(hze)).mean(axis=1), index=(t0e + np.arange(n_e)).astype(int))
    n_t = int(len(tmp) // hzt); t = pd.Series(tmp[:n_t * int(hzt)].reshape(n_t, int(hzt)).mean(axis=1), index=(t0t + np.arange(n_t)).astype(int))
    df = pd.DataFrame({'hr': h, 'acc_mean': a, 'acc_sd': s, 'eda': e, 'temp': t})
    try:
        ibi = pd.read_csv(sess / 'IBI.csv', skiprows=1, header=None).apply(pd.to_numeric, errors='coerce').dropna()
        ibi_t = t0h * 0 + float(open(sess / 'IBI.csv').readline().split(',')[0]) + ibi[0].to_numpy(); ibi_v = ibi[1].to_numpy()
        ib = np.column_stack([ibi_t, ibi_v])
    except Exception:
        ib = np.empty((0, 2))
    return df, ib

TABLES = {}
for pdir in sorted(EX.iterdir()):
    cf = CACHE / f'{pdir.name}.pkl'
    if cf.exists():
        TABLES[pdir.name] = pd.read_pickle(cf); continue
    frames, ibs = [], []
    for sess in sorted(pdir.iterdir()):
        df, ib = per_second(sess)
        if df is not None and len(df):
            frames.append(df); ibs.append(ib)
    sec = pd.concat(frames).sort_index()
    sec = sec[~sec.index.duplicated(keep='first')]
    ibi = np.vstack([i for i in ibs if len(i)]) if any(len(i) for i in ibs) else np.empty((0, 2))
    ibi = ibi[np.argsort(ibi[:, 0])] if len(ibi) else ibi
    pd.to_pickle({'sec': sec, 'ibi': ibi}, cf)
    TABLES[pdir.name] = {'sec': sec, 'ibi': ibi}
    print(pdir.name, 'seconds:', len(sec), '| hours:', round(len(sec) / 3600, 1))
print('nurses:', len(TABLES))

## Cell 2 - Stress events (the labels)
Survey times are local clock times. They line up with the sensor clock (UTC) when read as US Central time (322 of 358 events then have sensor data, against 237 for US Eastern). Events with no stress level are dropped.

In [ ]:
from zoneinfo import ZoneInfo
sv = pd.read_excel(BASE / 'SurveyResults.xlsx')
sv['ID'] = sv['ID'].astype(str)
sv = sv[sv['Stress level'].astype(str).isin(['0', '1', '2'])].copy()
sv['level'] = sv['Stress level'].astype(int)
tz = ZoneInfo('America/Chicago')
d = sv['date'].astype(str).str[:10]
sv['t_start'] = [pd.Timestamp(f'{a} {b}').to_pydatetime().replace(tzinfo=tz).timestamp() for a, b in zip(d, sv['Start time'].astype(str))]
sv['t_end'] = [pd.Timestamp(f'{a} {b}').to_pydatetime().replace(tzinfo=tz).timestamp() for a, b in zip(d, sv['End time'].astype(str))]
sv = sv[sv['t_end'] > sv['t_start']].reset_index(drop=True)
print('events with a stress level:', len(sv))
print(sv['level'].value_counts().sort_index().rename('events').to_string())
print('\nevents per nurse:'); print(sv.groupby('ID').size().to_string())

## Cell 3 - Personal baselines (no labels used)
Each nurse's own usual heart rate is the median heart rate in their low-movement minutes across all their recordings. This is what the app would learn over days. The population baseline is the median over nurses.

In [ ]:
def bvp_window(sec, a, b):
    w = sec.loc[a:b - 1]
    return w

base_rows = []
for pid, T in TABLES.items():
    sec = T['sec'][['hr', 'acc_sd']].dropna()
    g = sec.groupby(sec.index // 60)
    m = g.agg(hr=('hr', 'mean'), sd=('acc_sd', 'mean'), n=('hr', 'size'))
    m = m[m['n'] >= 40]
    calm = m[m['sd'] <= m['sd'].median()]
    base_rows.append({'ID': pid, 'base_hr': calm['hr'].median(), 'iqr_hr': max(calm['hr'].quantile(.75) - calm['hr'].quantile(.25), 2.0), 'calm_minutes': len(calm)})
BASE_T = pd.DataFrame(base_rows).set_index('ID')
POP_BASE = BASE_T['base_hr'].median(); POP_IQR = BASE_T['iqr_hr'].median()
print(BASE_T.round(1).to_string()); print('\npopulation baseline:', round(POP_BASE, 1), 'bpm')

## Cell 4 - Windows inside each reported event
Features: heart rate (mean, swing, slope), movement (level and variability), inter-beat-interval HRV where there are at least 20 intervals, EDA and temperature (lab ceiling). Up to 10 evenly spread windows per event.

In [ ]:
def hrv_feats(ibi_arr, a, b):
    if len(ibi_arr) == 0: return np.nan, np.nan
    m = (ibi_arr[:, 0] >= a) & (ibi_arr[:, 0] < b)
    v = ibi_arr[m, 1]; v = v[(v >= 0.33) & (v <= 1.5)]
    if len(v) < 20: return np.nan, np.nan
    dv = np.diff(v)
    return float(np.sqrt(np.mean(dv ** 2)) * 1000), float(v.std() * 1000)

rows = []
for ev_i, r in sv.iterrows():
    T = TABLES.get(r['ID']);
    if T is None: continue
    sec, ibi = T['sec'], T['ibi']
    starts = np.arange(int(r['t_start']), int(r['t_end']) - 60 + 1, 30)
    if len(starts) == 0: continue
    if len(starts) > 10: starts = starts[np.linspace(0, len(starts) - 1, 10).astype(int)]
    for a in starts:
        w = sec.loc[a:a + 59]
        h = w['hr'].dropna()
        if len(h) < 40: continue
        rm, sd = hrv_feats(ibi, a, a + 60)
        rows.append({'event': ev_i, 'ID': r['ID'], 'level': r['level'], 'start': a,
                     'hr_mean': h.mean(), 'hr_std': h.std(), 'hr_slope': np.polyfit(np.arange(len(h)), h.to_numpy(), 1)[0],
                     'acc_mean': w['acc_mean'].mean(), 'acc_sd': w['acc_sd'].mean(), 'rmssd': rm, 'sdnn': sd,
                     'eda': w['eda'].mean(), 'temp': w['temp'].mean()})
W = pd.DataFrame(rows)
W = W.join(BASE_T[['base_hr', 'iqr_hr']], on='ID')
W['hr_rel'] = (W['hr_mean'] - W['base_hr']) / W['iqr_hr']           # against the person's own usual
W['hr_pop'] = (W['hr_mean'] - POP_BASE) / POP_IQR                    # against the average working person
W.to_csv(OUT_DIR / 'nurses_event_windows.csv', index=False)
print('windows:', len(W), '| events:', W['event'].nunique(), '| nurses:', W['ID'].nunique())
print(W.groupby('level').agg(windows=('hr_mean', 'size'), events=('event', 'nunique'), hr=('hr_mean', 'mean'), hr_rel=('hr_rel', 'mean'), movement=('acc_sd', 'mean')).round(2).to_string())

## Cell 5 - Cross-validation helper
Participant-grouped, 10 fold layouts, out-of-fold predictions pooled per layout, shuffled-label control.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score, roc_auc_score, confusion_matrix

def make_model(name):
    if name == 'Majority': return DummyClassifier(strategy='prior')
    if name == 'LR':
        return Pipeline([('imp', SimpleImputer(strategy='median')), ('scale', StandardScaler()),
                         ('clf', LogisticRegression(max_iter=3000, class_weight='balanced', random_state=42))])
    return Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('clf', RandomForestClassifier(n_estimators=150, min_samples_leaf=8, class_weight='balanced', random_state=42, n_jobs=-1))])

REPEATS = 10
def run_cv(data, cols, label, model_name, shuffled=False):
    X, y, g = data[cols].to_numpy(), data[label].to_numpy(), data['ID'].to_numpy()
    rows, cms = [], []
    for s in range(REPEATS):
        yy = np.random.RandomState(700 + s).permutation(y) if shuffled else y
        prob = np.full(len(yy), np.nan)
        for tr, te in StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=s).split(X, yy, g):
            if len(np.unique(yy[tr])) < 2: prob[te] = yy[tr].mean(); continue
            prob[te] = make_model(model_name).fit(X[tr], yy[tr]).predict_proba(X[te])[:, 1]
        pred = (prob >= 0.5).astype(int)
        auc = 0.5 if model_name == 'Majority' else roc_auc_score(yy, prob)
        rows.append([accuracy_score(yy, pred), precision_score(yy, pred, zero_division=0), recall_score(yy, pred, zero_division=0),
                     f1_score(yy, pred, zero_division=0), balanced_accuracy_score(yy, pred), auc])
        cms.append(confusion_matrix(yy, pred, labels=[0, 1]))
    return np.array(rows), np.mean(cms, axis=0)

RES, CMS = [], {}
def evaluate(exp, data, cols, label, model_name):
    real, cm = run_cv(data, cols, label, model_name)
    shuf = real if model_name == 'Majority' else run_cv(data, cols, label, model_name, shuffled=True)[0]
    thr = max(shuf[:, 5].mean(), 0.5) + 2 * shuf[:, 5].std()
    RES.append({'experiment': exp, 'model': model_name, 'rows': len(data), 'people': data['ID'].nunique(),
                'accuracy': real[:, 0].mean(), 'precision': real[:, 1].mean(), 'recall': real[:, 2].mean(), 'F1': real[:, 3].mean(),
                'balanced acc': real[:, 4].mean(), 'AUC': real[:, 5].mean(), 'AUC std': real[:, 5].std(), 'shuffled AUC': shuf[:, 5].mean(),
                'clearly above chance?': 'n/a' if model_name == 'Majority' else ('YES' if real[:, 5].mean() > thr else 'no')})
    CMS[(exp, model_name)] = cm
print('helper ready')

## Cell 6 - Experiments
Main: high stress (2) vs no stress (0). Second: any stress (1 or 2) vs no stress.

In [ ]:
MOVE = ['acc_mean', 'acc_sd']
SETS = [
    ('A heart rate + movement (raw)', ['hr_mean', 'hr_std', 'hr_slope'] + MOVE),
    ('A2 heart rate vs average person + movement', ['hr_pop', 'hr_std', 'hr_slope'] + MOVE),
    ('B heart rate vs OWN usual + movement', ['hr_rel', 'hr_std', 'hr_slope'] + MOVE),
    ('C + heart-rate variability', ['hr_rel', 'hr_std', 'hr_slope', 'rmssd', 'sdnn'] + MOVE),
    ('D + EDA and temperature (lab ceiling)', ['hr_rel', 'hr_std', 'hr_slope', 'rmssd', 'sdnn', 'eda', 'temp'] + MOVE),
    ('E own usual only (no movement)', ['hr_rel', 'hr_std', 'hr_slope']),
]
main = W[W['level'].isin([0, 2])].copy(); main['y'] = (main['level'] == 2).astype(int)
second = W.copy(); second['y'] = (second['level'] >= 1).astype(int)
print('main target classes (windows):', main['y'].value_counts().to_dict(), '| nurses:', main['ID'].nunique())
for name, cols in SETS:
    for m in ['Majority', 'LR', 'RF']:
        evaluate('M ' + name, main, cols, 'y', m)
for name, cols in [SETS[0], SETS[2]]:
    for m in ['Majority', 'RF']:
        evaluate('S ' + name + ' | any stress vs none', second, cols, 'y', m)
res = pd.DataFrame(RES)
show = ['experiment', 'model', 'rows', 'people', 'accuracy', 'precision', 'recall', 'F1', 'balanced acc', 'AUC', 'shuffled AUC', 'clearly above chance?']
print(res[show].round(3).to_string(index=False))
res.round(4).to_csv(OUT_DIR / 'nurses_stress_results.csv', index=False)

## Cell 7 - Confusion matrix and figure

In [ ]:
import matplotlib.pyplot as plt
cand = res[(res['experiment'].str.startswith('M B')) & (res['model'] != 'Majority')].sort_values('AUC', ascending=False).iloc[0]
cm = CMS[(cand['experiment'], cand['model'])]
print(f"{cand['experiment']} / {cand['model']}: AUC {cand['AUC']:.2f}, balanced acc {cand['balanced acc']:.2f}, F1 {cand['F1']:.2f}")
print(pd.DataFrame(cm.round(0), index=['true no stress (0)', 'true high stress (2)'], columns=['pred no stress', 'pred high stress']).to_string())
rf = res[res['model'] == 'RF'].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(10, 4.5)); y = np.arange(len(rf))
ax.barh(y, rf['AUC'], xerr=rf['AUC std'], color='tab:blue', alpha=0.85, label='real labels')
ax.scatter(rf['shuffled AUC'], y, color='k', zorder=3, label='shuffled labels')
ax.axvline(0.5, color='gray', linestyle='--', linewidth=1)
ax.set_yticks(y); ax.set_yticklabels([e[:62] for e in rf['experiment']], fontsize=7); ax.invert_yaxis()
ax.set_xlim(0.3, 1.0); ax.set_xlabel('ROC-AUC (Random Forest, nurses held out)'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT_DIR / 'nurses_stress_results.png', dpi=120); plt.show()

## Cell 8 - How to read this (report text)
- **A vs B:** does comparing heart rate with the person's OWN usual beat comparing it with the average working person? If yes, the personal baseline idea is supported in real life.
- **With and without movement (B vs E):** does knowing about movement stop exercise-like heart-rate rises being read as stress?
- **D is a ceiling**, not deployable (phones and most watches give no EDA).
- A model counts only if `clearly above chance? = YES` and it beats the Majority row.
- **Limits to state**: 15 nurses (not students), labels are the person's own report after an event, events are chosen by the study (stress-flagged), real-life noise, wrist sensor, US hospital during COVID. The output is a **physiological stress load**, not a diagnosis.